In [ ]:
%load_ext autoreload
%autoreload 2

import mofaflex as mfl
import matplotlib.pyplot as plt
from plotnine import *
import pandas as pd
import numpy as np
import anndata as ad
import scanpy as sc
import pickle as pkl
import h5py
import utils
from io import StringIO
import requests
import networkx as nx
import h5py

## Load Data

In [ ]:
celltype = "A549"

adata = ad.read_h5ad(f"data/anndata.h5ad", backed="r")
adata = ad.AnnData(obs=adata.obs, var=adata.var, varm=adata.varm, obsm=adata.obsm)
adata = adata[(adata.obs.cell_type==celltype) & (adata.obs.pathway != "INS")]
adata.obs_names_make_unique()

## Get Factors and Loadings

### MOFAFLEX

In [ ]:
mfl_model = mfl.MOFAFLEX.load(f"models/by_cellline/mfl_A549.h5")
mfl_factors = mfl_model.get_factors("pandas")[celltype]
mfl_loadings = mfl_model.get_weights("pandas")["RNA"]

In [ ]:
# subset anndata to used genes
adata = adata[:, mfl_model.feature_names["RNA"]]

mask = pd.read_csv("hallmark_mask.csv", index_col=0)
mask = mask.loc[adata.var_names]
mask = mask.loc[:, mask.sum(axis=0) >= 5]

adata.varm["annotations"] = mask

### Spectra

In [ ]:
spectra_dict = pkl.load(open(f"models/spectra_{celltype}.pkl", "rb"))

spectra_factors = pd.DataFrame(spectra_dict["cell_scores"], index=spectra_dict["obs_names"])
spectra_loadings = pd.DataFrame(spectra_dict["factors"], columns=spectra_dict["var_names"])

spectra_factors.columns = [f"Factor {i}" for i in range(1, spectra_factors.shape[1]+1)]
spectra_loadings.index = [f"Factor {i}" for i in range(1, spectra_factors.shape[1]+1)]

# assign gene sets to factors based on overlap of Spectra factor markers
spectra_gs_assignment = utils.assign_markers_to_gene_sets(
    spectra_dict["SPECTRA_markers"], adata.varm["annotations"]
)
# if duplicates are present, keep assignment with highest overlap
spectra_gs_assignment = spectra_gs_assignment.sort_values('overlap_coefficient', ascending=False).drop_duplicates('gene_set').sort_index()

# remove unassigned factors
spectra_gs_assignment = spectra_gs_assignment[spectra_gs_assignment.overlap_coefficient > 0.2]

spectra_factors = spectra_factors[spectra_gs_assignment.factor]
spectra_factors.columns = spectra_gs_assignment.gene_set

spectra_loadings = spectra_loadings.loc[spectra_gs_assignment.factor]
spectra_loadings.index = spectra_gs_assignment.gene_set

### ExpiMap

In [ ]:
expimap_dict = pkl.load(open(f"models/expimap_{celltype}.pkl", "rb"))

expimap_factors = pd.DataFrame(expimap_dict["latents"])
expimap_factors.columns = mask.columns
expimap_factors.index = expimap_dict["obs_names"]

expimap_loadings = pd.DataFrame(expimap_dict["weights"], index=mask.columns, columns=expimap_dict["var_names"])

## Prettify Factor Names

In [ ]:
hallmark_pretty = {
    "HALLMARK_ADIPOGENESIS": "Adipogenesis",
    "HALLMARK_ALLOGRAFT_REJECTION": "Allograft Rejection",
    "HALLMARK_ANDROGEN_RESPONSE": "Androgen Response",
    "HALLMARK_ANGIOGENESIS": "Angiogenesis",
    "HALLMARK_APICAL_JUNCTION": "Apical Junction",
    "HALLMARK_APICAL_SURFACE": "Apical Surface",
    "HALLMARK_APOPTOSIS": "Apoptosis",
    "HALLMARK_BILE_ACID_METABOLISM": "Bile Acid Metabolism",
    "HALLMARK_CHOLESTEROL_HOMEOSTASIS": "Cholesterol Homeostasis",
    "HALLMARK_COAGULATION": "Coagulation",
    "HALLMARK_COMPLEMENT": "Complement",
    "HALLMARK_DNA_REPAIR": "DNA Repair",
    "HALLMARK_E2F_TARGETS": "E2F Targets",
    "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION": "Epithelial-Mesenchymal Transition",
    "HALLMARK_ESTROGEN_RESPONSE_EARLY": "Estrogen Response (Early)",
    "HALLMARK_ESTROGEN_RESPONSE_LATE": "Estrogen Response (Late)",
    "HALLMARK_FATTY_ACID_METABOLISM": "Fatty Acid Metabolism",
    "HALLMARK_G2M_CHECKPOINT": "G2M Checkpoint",
    "HALLMARK_GLYCOLYSIS": "Glycolysis",
    "HALLMARK_HEDGEHOG_SIGNALING": "Hedgehog Signaling",
    "HALLMARK_HEME_METABOLISM": "Heme Metabolism",
    "HALLMARK_HYPOXIA": "Hypoxia",
    "HALLMARK_IL2_STAT5_SIGNALING": "IL2/STAT5 Signaling",
    "HALLMARK_IL6_JAK_STAT3_SIGNALING": "IL6/JAK/STAT3 Signaling",
    "HALLMARK_INFLAMMATORY_RESPONSE": "Inflammatory Response",
    "HALLMARK_INTERFERON_ALPHA_RESPONSE": "Interferon Alpha Response",
    "HALLMARK_INTERFERON_GAMMA_RESPONSE": "Interferon Gamma Response",
    "HALLMARK_KRAS_SIGNALING_DN": "KRAS Signaling (Down)",
    "HALLMARK_KRAS_SIGNALING_UP": "KRAS Signaling (Up)",
    "HALLMARK_MITOTIC_SPINDLE": "Mitotic Spindle",
    "HALLMARK_MTORC1_SIGNALING": "mTORC1 Signaling",
    "HALLMARK_MYC_TARGETS_V1": "MYC Targets V1",
    "HALLMARK_MYC_TARGETS_V2": "MYC Targets V2",
    "HALLMARK_MYOGENESIS": "Myogenesis",
    "HALLMARK_NOTCH_SIGNALING": "Notch Signaling",
    "HALLMARK_OXIDATIVE_PHOSPHORYLATION": "Oxidative Phosphorylation",
    "HALLMARK_P53_PATHWAY": "p53 Pathway",
    "HALLMARK_PANCREAS_BETA_CELLS": "Pancreas Beta Cells",
    "HALLMARK_PEROXISOME": "Peroxisome",
    "HALLMARK_PI3K_AKT_MTOR_SIGNALING": "PI3K/AKT/mTOR Signaling",
    "HALLMARK_PROTEIN_SECRETION": "Protein Secretion",
    "HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY": "Reactive Oxygen Species",
    "HALLMARK_SPERMATOGENESIS": "Spermatogenesis",
    "HALLMARK_TGF_BETA_SIGNALING": "TGF-beta Signaling",
    "HALLMARK_TNFA_SIGNALING_VIA_NFKB": "TNFa Signaling via NF-kB",
    "HALLMARK_UNFOLDED_PROTEIN_RESPONSE": "Unfolded Protein Response",
    "HALLMARK_UV_RESPONSE_DN": "UV Response (Down)",
    "HALLMARK_UV_RESPONSE_UP": "UV Response (Up)",
    "HALLMARK_WNT_BETA_CATENIN_SIGNALING": "WNT/beta-Catenin Signaling",
    "HALLMARK_XENOBIOTIC_METABOLISM": "Xenobiotic Metabolism",
}

## Explained Variance

In [ ]:
n_top_factors = 10

df_r2 = (
    mfl_model.get_r2()[celltype]
    .reset_index(names="Factor")
    .replace(hallmark_pretty)
    .sort_values("RNA", ascending=False)
)

# Remove uninformed factors
df_r2 = df_r2[~df_r2.Factor.str.startswith("Factor")]

# split in two groups based on explained variance, aggregate second (lower) group
top = df_r2.iloc[:n_top_factors]
bottom = pd.DataFrame({"Factor": ["Remaining Factors"], "RNA": [df_r2["RNA"].iloc[n_top_factors:].sum()]})
df_r2 = pd.concat([top, bottom], ignore_index=True)

df_r2["Factor"] = pd.Categorical(df_r2["Factor"], categories=df_r2["Factor"][::-1], ordered=True)
df_r2["in_group"] = df_r2.Factor.isin(["TNFa Signaling via NF-kB", "Interferon Gamma Response", "Interferon Alpha Response"])

plot = (
    ggplot(df_r2, aes(x="Factor", y="RNA", fill="in_group"))
    + geom_bar(stat="identity")
    + coord_flip()
    + theme_bw()
    + labs(y="R2", x="")
    + theme(figure_size=(3, 3), legend_position="none", axis_text_x=element_text(rotation=90))
    + scale_fill_manual({False: "grey", True: "green"})
)
plot.show()

## Prediction of Pathway Stimulation

In [ ]:
def make_auroc_df(factors, pathway_obs, method_name, subsample_size=1000):
    df = (
        utils.compute_auroc_matrix(factors, pathway_obs, subsample_size=subsample_size)
        .reset_index(names="Factor")
        .melt(id_vars="Factor", var_name="Pathway", value_name="AUROC")
        .replace(hallmark_pretty)
    )
    df = df[~df.Factor.str.startswith("Factor")]
    df["Method"] = method_name
    return df

dfs = [
    make_auroc_df(mfl_factors, adata.obs["pathway"], "MOFA-FLEX"),
    make_auroc_df(spectra_factors, adata.obs["pathway"], "Spectra"),
    make_auroc_df(expimap_factors, adata.obs["pathway"], "ExpiMap"),
]

# Collect union of top-2 factors per pathway across all methods
top_factors = set()
for df in dfs:
    method_top = (
        df.groupby("Pathway")
        .apply(lambda g: g.nlargest(2, "AUROC"))
        .reset_index(drop=True)
    ).Factor.unique()
    top_factors.update(method_top)

# Subset all methods to the same shared factor set
plot_df = pd.concat(
    [df[df.Factor.isin(top_factors)] for df in dfs],
    ignore_index=True,
)
plot_df["Method"] = pd.Categorical(plot_df["Method"], categories=["MOFA-FLEX", "Spectra", "ExpiMap"], ordered=True)

plot = (
    ggplot(plot_df, aes(x="Pathway", y="Factor", fill="AUROC"))
    + geom_tile()
    + scale_fill_gradientn(colors=["#f7fbff", "#2171b5"], limits=[0.5, 1], name="AUROC")
    + facet_wrap("Method", ncol=3)
    + theme_bw()
    + theme(
        axis_text_x=element_text(rotation=90, hjust=0.5, size=9),
        axis_text_y=element_text(size=9),
        strip_text=element_text(size=8, face="bold"),
        panel_grid=element_blank(),
        figure_size=(5.5, 4),
    )
    + labs(x="Pathway", y="")
)
plot.show()

## UMAPs

In [ ]:
def make_factors_adata(factors, obs):
    a = ad.AnnData(factors.loc[obs.index])
    a.obs = obs
    sc.pp.neighbors(a)
    sc.tl.umap(a)
    return a

mfl_factors_adata = make_factors_adata(mfl_factors, adata.obs)
spectra_factors_adata = make_factors_adata(spectra_factors, adata.obs)
expimap_factors_adata = make_factors_adata(expimap_factors, adata.obs)

In [ ]:
def plot_umap(factors_adata):
    umap_coords = pd.DataFrame(
        factors_adata.obsm[f"X_umap"], columns=["UMAP1", "UMAP2"], index=factors_adata.obs_names
    ).join(factors_adata.obs[["pathway", "Batch_info"]]).sample(100000)

    plots = []
    for color, label in [("pathway", "Pathway"), ("Batch_info", "Batch")]:
        plot = (
            ggplot(umap_coords, aes(x="UMAP2", y="UMAP1", color=color))
            + geom_point(size=0.1, alpha=0.1, raster=True)
            + theme_bw()
            + labs(title=f"", color=label)
            + guides(color=guide_legend(override_aes={"size": 3, "alpha": 1}))
            + theme(
                figure_size=(3.5, 3),
                axis_text_x=element_blank(),
                axis_text_y=element_blank(),
                axis_ticks_x=element_blank(),
                axis_ticks_y=element_blank(),
            )
            + coord_equal()
        )
        plots.append(plot)
    return plots
    
mfl_umaps = plot_umap(mfl_factors_adata)
spectra_umaps = plot_umap(spectra_factors_adata)
expimap_umaps = plot_umap(expimap_factors_adata)

## Factor Scores by Pathway

In [ ]:
plot_factors = [
    "TNFa Signaling via NF-kB",
    "Interferon Gamma Response",
    "Interferon Alpha Response",
    "G2M Checkpoint",
    "Epithelial-Mesenchymal Transition",
    ]

plot_df = (
    mfl_factors
    .join(adata.obs[["pathway"]])
    .reset_index(names="cell_id")
    .melt(id_vars=["cell_id", "pathway"], var_name="factor", value_name="score")
)
plot_df["factor"] = plot_df["factor"].replace(hallmark_pretty)
plot_df = plot_df[plot_df.factor.isin(plot_factors)]
plot_df.factor = pd.Categorical(plot_df.factor, categories=plot_factors[::-1], ordered=True)

plot = (
    ggplot(plot_df, aes(x="factor", y="score", fill="pathway"))
    + geom_boxplot(outlier_size=0.1)
    + theme_bw()
    + theme(figure_size=(5, 4))
    + labs(x="Factor", y="Factor Score", fill="Pathway")
    + coord_flip()
)
plot.show()

## Top Weights

In [ ]:
plot = mfl.pl.weights(mfl_model, factors=["HALLMARK_INTERFERON_ALPHA_RESPONSE", "HALLMARK_G2M_CHECKPOINT", "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION"], n_features=15)
plot += facet_grid(cols="factor", labeller=labeller(cols=hallmark_pretty))
plot += theme_bw()
plot += theme(figure_size=(10, 3))
plot += guides(color=guide_legend(override_aes={"size": 3}))
plot.show()

## Gene Set Relationships for Inferred Genes using STRING DB

In [ ]:
STRING_API = "https://string-db.org/api"
SPECIES = 9606  # human
SCORE_THRESHOLD = 500  # 0–1000; 700 = high confidence, 900 = highest

def map_to_string_ids(genes, species=SPECIES):
    url = f"{STRING_API}/tsv/get_string_ids"
    params = {
        "identifiers": "\r".join(genes),
        "species": species,
        "limit": 1,
        "echo_query": 1,
    }
    r = requests.post(url, data=params)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), sep="\t")
    # Keep first hit per query
    return df.drop_duplicates(subset="queryItem").set_index("queryItem")["stringId"].to_dict()


def get_network(string_ids, species=SPECIES, score_threshold=SCORE_THRESHOLD):
    url = f"{STRING_API}/tsv/network"
    params = {
        "identifiers": "\r".join(string_ids),
        "species": species,
        "required_score": score_threshold,
    }
    r = requests.post(url, data=params)
    r.raise_for_status()
    return pd.read_csv(StringIO(r.text), sep="\t")


def score_query_genes(network: pd.DataFrame, query_symbols, hallmark_symbols) -> pd.DataFrame:
    records = []
    for q in query_symbols:
        # Edges where one endpoint is the query and the other is in hallmark
        mask = (
            ((network["preferredName_A"] == q) & (network["preferredName_B"].isin(hallmark_symbols))) |
            ((network["preferredName_B"] == q) & (network["preferredName_A"].isin(hallmark_symbols)))
        )
        edges = network[mask]
        if edges.empty:
            records.append({
                "gene": q, "n_edges": 0, "mean_score": np.nan,
                "sum_score": 0.0, "max_score": np.nan, "neighbours": ""
            })
            continue

        # Identify the hallmark neighbour for each edge
        neighbours = np.where(
            edges["preferredName_A"] == q,
            edges["preferredName_B"],
            edges["preferredName_A"],
        )
        scores = edges["score"].values  # already 0–1 in tsv endpoint

        records.append({
            "gene": q,
            "n_edges": len(edges),
            "mean_score": scores.mean(),
            "sum_score": scores.sum(),
            "max_score": scores.max(),
            "neighbours": ", ".join(sorted(set(neighbours))),
        })

    return pd.DataFrame(records)

def network_enrichment(query_ids, background_ids, species=SPECIES):
    """
    Tests whether the query gene set is significantly connected to a
    background set (here, the hallmark genes act as the background context).
    """
    url = f"{STRING_API}/tsv/ppi_enrichment"
    params = {
        "identifiers": "\r".join(query_ids + background_ids),
        "species": species,
    }
    r = requests.post(url, data=params)
    r.raise_for_status()
    return pd.read_csv(StringIO(r.text), sep="\t")

In [ ]:
string_results = {}
n_query = 15

for geneset in ["HALLMARK_INTERFERON_ALPHA_RESPONSE", "HALLMARK_G2M_CHECKPOINT", "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION"]:
    # get hallmark genes
    hallmark_genes = adata.varm["annotations"][adata.varm["annotations"][geneset]].index.tolist()

    # get top n_query genes with highest loadings which are not in Hallmark
    query_genes = mfl_model.get_weights()["RNA"].T[geneset].sort_values(ascending=False).index.tolist()
    query_genes = [x for x in query_genes[:n_query] if x not in hallmark_genes]

    # map gene symbols to STRING identifiers
    hallmark_map = map_to_string_ids(hallmark_genes)
    query_map = map_to_string_ids(query_genes)

    print(f"Mapped {len(hallmark_map)}/{len(hallmark_genes)} hallmark genes")
    print(f"Mapped {len(query_map)}/{len(query_genes)} query genes")

    # get protein-protein interaction network for relevant genes
    all_ids = list(set(hallmark_map.values()) | set(query_map.values()))
    network = get_network(all_ids)

    print(f"\nFetched {len(network)} edges at score >= {SCORE_THRESHOLD/1000:.2f}")

    hallmark_symbols = set(hallmark_genes)
    query_symbols = set(query_genes)

    # get scores for query genes
    results = score_query_genes(network, query_symbols, hallmark_symbols)

    # Rank by sum_score (rewards both number and strength of connections)
    results = results.sort_values("sum_score", ascending=False).reset_index(drop=True)

    enrichment = network_enrichment(
        list(query_map.values()), list(hallmark_map.values())
    )

    string_results[geneset] = {}
    string_results[geneset]["results"] = results
    string_results[geneset]["enrichment"] = enrichment

In [ ]:
df_plot = string_results["HALLMARK_G2M_CHECKPOINT"]["results"]
df_plot["gene"] = pd.Categorical(df_plot["gene"], categories=df_plot.gene, ordered=True)

plot = (
    ggplot(df_plot, aes(x="gene", y="n_edges"))
    + geom_bar(stat="identity")
    + theme_bw()
    + scale_y_continuous(expand=(0, 0, 0.04, 0))
    + labs(x="Inferred Gene", y="# Edges", title="G2M Checkpoint Factor")
    + theme(figure_size=(5, 3), axis_text_x=element_text(rotation=45))
)
plot.show()


In [ ]:
df_plot = string_results["HALLMARK_INTERFERON_ALPHA_RESPONSE"]["results"]
df_plot["gene"] = pd.Categorical(df_plot["gene"], categories=df_plot.gene, ordered=True)

plot = (
    ggplot(df_plot, aes(x="gene", y="n_edges"))
    + geom_bar(stat="identity")
    + theme_bw()
    + scale_y_continuous(expand=(0, 0, 0.04, 0))
    + labs(x="Inferred Gene", y="# Edges", title="Interferon Alpha Response Factor")
    + theme(figure_size=(5, 3), axis_text_x=element_text(rotation=45))
)
plot.show()

In [ ]:
df_plot = string_results["HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION"]["results"]
df_plot["gene"] = pd.Categorical(df_plot["gene"], categories=df_plot.gene, ordered=True)

plot = (
    ggplot(df_plot, aes(x="gene", y="n_edges"))
    + geom_bar(stat="identity")
    + theme_bw()
    + scale_y_continuous(expand=(0, 0, 0.04, 0))
    + labs(x="Inferred Gene", y="# Edges", title="Epithelial Mesenchymal Transition Factor")
    + theme(figure_size=(5, 3), axis_text_x=element_text(rotation=45))
)
plot.show()

## Gene Weight Overlap Visualizations

In [ ]:
expimap_factor_name = "HALLMARK_INTERFERON_ALPHA_RESPONSE"

expimap_ifna = expimap_loadings.T[expimap_factor_name].sort_values(ascending=False).reset_index()
expimap_ifna.columns = ["Gene", "Weight"]

hallmark_ifna = mask[mask[expimap_factor_name]].index.tolist()

N_TOP = 15  # top genes per method to consider

hallmark_set = set(hallmark_ifna)

def make_lollipop_df(df, method_name, factor_name):
    d = df.head(N_TOP).copy()
    d["Rank"] = range(1, len(d) + 1)
    d["Method"] = f"{method_name}\n{hallmark_pretty[factor_name]}"
    d["In Hallmark"] = d["Gene"].isin(hallmark_set)
    d["Nudge"] = d["Weight"].max() * 0.03
    return d

lollipop_df = make_lollipop_df(expimap_ifna, "ExpiMap", expimap_factor_name)

lollipop_df["In Hallmark"] = lollipop_df["In Hallmark"].map({False: "Inferred", True: "Annotated"})

plot = (
    ggplot(lollipop_df, aes(x="Rank", y="Weight", color="In Hallmark", label="Gene"))
    + geom_segment(aes(xend="Rank", yend=0), size=0.6, alpha=0.7)
    + geom_point(size=2.5)
    + geom_text(aes(y="Weight + Nudge"), size=6, ha="left", angle=0)
    + scale_color_manual({"Annotated": "black", "Inferred": "red"}, name="")
    + scale_x_reverse()
    + scale_y_continuous(expand=(0.05, 0, 0.2, 0))
    + coord_flip()
    + theme_bw()
    + theme(figure_size=(5, 4), legend_position="right",
            axis_text_y=element_text(size=7))
    + labs(x="Rank", y="Weight", title="Expimap Interferon Alpha Response Factor", color="")
)
plot.show()